# TP3 — Logistic Regression on Arduino Uno

Run all. The public smoke dataset is downloaded if missing. The notebook trains the two-sensor model and generates src/main.cpp; download that sketch to build in PlatformIO/Wokwi.

This notebook contains the full Python solution. Board projects also require the included firmware. Read EXPLICATION_AR.md for the detailed Arabic explanation.

[Assignment](https://github.com/ahmed3991/AIoT-2025/tree/main/TP3)

In [1]:
# Install missing dependencies before importing the model libraries.
import importlib.util
import importlib.metadata
import subprocess
import sys

required = {'pandas': 'pandas', 'sklearn': 'scikit-learn', 'numpy': 'numpy'}
missing = []
for module_name, package_spec in required.items():
    try:
        found = importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        found = False
    if not found:
        missing.append(package_spec)
    elif '==' in package_spec:
        distribution, expected = package_spec.split('==')
        if importlib.metadata.version(distribution) != expected:
            missing.append(package_spec)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Dependencies ready.')


Dependencies ready.


## 1. Imports and settings

In [2]:
"""TP3: learn a two-sensor model and export a complete Arduino Uno sketch."""
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / "data" / "smoke_detection_iot.csv"



## 2. Load data and keep the two sensors actually available on the Uno

In [3]:
if not DATA_PATH.exists():
    from io import BytesIO
    from urllib.request import urlopen
    from zipfile import ZipFile
    uploaded = ROOT / "smoke_detection_iot.csv"
    kaggle = list(Path("/kaggle/input").rglob("smoke_detection_iot.csv"))
    if uploaded.exists():
        DATA_PATH = uploaded
    elif kaggle:
        DATA_PATH = kaggle[0]
    else:
        url = "https://www.kaggle.com/api/v1/datasets/download/deepcontractor/smoke-detection-dataset"
        with urlopen(url, timeout=60) as response:
            with ZipFile(BytesIO(response.read())) as archive:
                name = next(name for name in archive.namelist() if name.endswith("smoke_detection_iot.csv"))
                DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
                DATA_PATH.write_bytes(archive.read(name))
features = ["Temperature[C]", "Humidity[%]"]
data = pd.read_csv(DATA_PATH)[features + ["Fire Alarm"]].dropna()
X_train, X_test, y_train, y_test = train_test_split(
    data[features], data["Fire Alarm"], test_size=0.2, random_state=42,
    stratify=data["Fire Alarm"],
)



## 3. Train the two-feature pipeline and extract mean, scale, weights, bias

In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
metrics = {"accuracy": accuracy_score(y_test, predictions),
           "f1": f1_score(y_test, predictions), "recall": recall_score(y_test, predictions)}
scaler = pipeline["scaler"]
estimator = pipeline["model"]
parameters = {
    "features": features,
    "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
    "weights": estimator.coef_[0].tolist(), "bias": float(estimator.intercept_[0]),
}
(OUT / "parameters.json").write_text(json.dumps(parameters, indent=2), encoding="utf-8")
with (OUT / "two_sensor_pipeline.pkl").open("wb") as file:
    pickle.dump(pipeline, file, protocol=pickle.HIGHEST_PROTOCOL)
print("Metrics:", metrics)
print("Exported parameters:", parameters)



Metrics: {'accuracy': 0.7711160785566022, 'f1': 0.8589838178151591, 'recall': 0.9755334599486091}
Exported parameters: {'features': ['Temperature[C]', 'Humidity[%]'], 'mean': [15.953115080632285, 48.57430185214754], 'scale': [14.358038477662703, 8.821187741623913], 'weights': [-0.2086760553936151, 0.9679781403658625], 'bias': 0.9693804113331901}


## 4. Verify the manual float32 formula against the Python pipeline

In [5]:
raw = X_test.to_numpy(dtype=np.float32)
means = np.asarray(parameters["mean"], dtype=np.float32)
scales = np.asarray(parameters["scale"], dtype=np.float32)
weights = np.asarray(parameters["weights"], dtype=np.float32)
z = ((raw - means) / scales) @ weights + np.float32(parameters["bias"])
manual_probability = 1 / (1 + np.exp(-z))
python_probability = pipeline.predict_proba(X_test)[:, 1]
max_error = float(np.max(np.abs(manual_probability - python_probability)))
np.testing.assert_allclose(manual_probability, python_probability, atol=1e-6, rtol=1e-5)
assert np.array_equal((z > 0).astype(int), predictions)
print("Manual formula matches the pipeline. Maximum probability error:", max_error)



Manual formula matches the pipeline. Maximum probability error: 1.2235158747264663e-07


## 5. Generate the complete required src/main.cpp

In [6]:
def cpp_array(values):
    return ", ".join(f"{value:.9e}f" for value in values)

firmware = """// TP3: trained two-sensor Logistic Regression, generated by tp3.py.
#include <Arduino.h>
#include <DHT.h>
#include <math.h>

DHT dht(2, DHT22);
const int N_FEATURES = 2; // Temperature, then Humidity
const float MEAN[2] = {@MEAN@};
const float STD[2] = {@STD@};
const float WEIGHTS[2] = {@WEIGHTS@};
const float BIAS = @BIAS@;

float standardize(float value, int index) {
  return (value - MEAN[index]) / STD[index];
}

float sigmoid(float z) {
  // Numerically stable even for large positive/negative inputs.
  if (z >= 0) return 1.0f / (1.0f + expf(-z));
  const float e = expf(z);
  return e / (1.0f + e);
}

float predict(const float scaled[]) {
  float z = BIAS;
  for (int i = 0; i < N_FEATURES; ++i) z += WEIGHTS[i] * scaled[i];
  return sigmoid(z);
}

void setup() {
  Serial.begin(9600);
  dht.begin();
  pinMode(LED_BUILTIN, OUTPUT);
}

void loop() {
  delay(2000); // DHT22 needs at least two seconds between readings.
  const float temperature = dht.readTemperature();
  const float humidity = dht.readHumidity();
  if (isnan(temperature) || isnan(humidity)) {
    Serial.println(F("DHT read failed"));
    return;
  }
  const float scaled[2] = {standardize(temperature, 0), standardize(humidity, 1)};
  const unsigned long start = micros();
  const float probability = predict(scaled);
  const unsigned long inferenceUs = micros() - start;
  const int prediction = probability > 0.5f ? 1 : 0;
  digitalWrite(LED_BUILTIN, prediction ? HIGH : LOW);
  Serial.print(F("Temperature=")); Serial.print(temperature);
  Serial.print(F(" Humidity=")); Serial.print(humidity);
  Serial.print(F(" Probability=")); Serial.print(probability, 6);
  Serial.print(F(" Class=")); Serial.print(prediction);
  Serial.print(F(" Inference_us=")); Serial.println(inferenceUs);
}
"""
for key, values in [("MEAN", parameters["mean"]), ("STD", parameters["scale"]),
                    ("WEIGHTS", parameters["weights"])]:
    firmware = firmware.replace(f"@{key}@", cpp_array(values))
firmware = firmware.replace("@BIAS@", f"{parameters['bias']:.9e}f")
(ROOT / "src").mkdir(exist_ok=True)
(ROOT / "src" / "main.cpp").write_text(firmware, encoding="utf-8")
report = f"""# TP3 report

The DHT22 supplies temperature and humidity only. TP2's 12-feature pipeline
cannot accept these two inputs without the missing sensors. A new two-feature
pipeline was therefore trained on the same smoke dataset, 80/20 stratified split,
seed 42. This follows TP3's stated two-input requirement rather than inventing
the other ten readings. This model is different from TP2's full model.

Accuracy: {metrics['accuracy']:.6f}; F1: {metrics['f1']:.6f}; recall: {metrics['recall']:.6f}.
Mean, scale, coefficients and intercept are saved in parameters.json and embedded
as constants directly in src/main.cpp. The float32 manual inference formula
matches sklearn across all {len(X_test)} test readings, maximum probability error
{max_error:.3e}. Seven floats describe the two means, two scales, two weights
and one bias (about 28 bytes), excluding program and sensor-library memory.

The complete sketch standardizes Temperature then Humidity, computes the weighted
sum and sigmoid, prints probability/class, and controls the Uno's built-in LED.
PlatformIO builds use an Arduino Uno and DHT22 connected to digital pin 2.
The sketch prints on-device inference microseconds when run. Desktop numerical
agreement does not certify real-world fire detection. Hardware/Wokwi observations
must be recorded separately; no fabricated board measurements are included.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP3/TP3_Logistic_Arduino.md
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Generated", ROOT / "src" / "main.cpp")


Generated C:\Users\chikh\Desktop\UNIV\ghanabzia\TP3-Ghanabzia\src\main.cpp


In [7]:
from IPython.display import display, Markdown
display(Markdown((OUT).joinpath('REPORT.md').read_text(encoding='utf-8')))


# TP3 report

The DHT22 supplies temperature and humidity only. TP2's 12-feature pipeline
cannot accept these two inputs without the missing sensors. A new two-feature
pipeline was therefore trained on the same smoke dataset, 80/20 stratified split,
seed 42. This follows TP3's stated two-input requirement rather than inventing
the other ten readings. This model is different from TP2's full model.

Accuracy: 0.771116; F1: 0.858984; recall: 0.975533.
Mean, scale, coefficients and intercept are saved in parameters.json and embedded
as constants directly in src/main.cpp. The float32 manual inference formula
matches sklearn across all 12526 test readings, maximum probability error
1.224e-07. Seven floats describe the two means, two scales, two weights
and one bias (about 28 bytes), excluding program and sensor-library memory.

The complete sketch standardizes Temperature then Humidity, computes the weighted
sum and sigmoid, prints probability/class, and controls the Uno's built-in LED.
PlatformIO builds use an Arduino Uno and DHT22 connected to digital pin 2.
The sketch prints on-device inference microseconds when run. Desktop numerical
agreement does not certify real-world fire detection. Hardware/Wokwi observations
must be recorded separately; no fabricated board measurements are included.

Source: https://github.com/ahmed3991/AIoT-2025/blob/main/TP3/TP3_Logistic_Arduino.md
